In [ ]:
import re
import json
from pathlib import Path
from yarn_utils import YARNGraph
from grewpy import Graph, GRS
import itertools
import networkx as nx

# Load Data

In [406]:
# FOLDER_PATH = "annotations/FRACAS_12032026/"
# FILE = "105h.yarn.json"

# FOLDER_PATH = "annotations/"
# FILE = "1.yarn.json"

# FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
# FILE = "106h.yarn.json"

FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
FILE = "89p.yarn.json"

In [407]:
with open(FOLDER_PATH + FILE) as f:
    yarn_graph_json = json.load(f)

yarn_graph = YARNGraph(yarn_graph_json)
yarn_grew = yarn_graph.grew()

In [408]:
with open('output.json', 'w') as f:
    json.dump(yarn_grew, f)

In [409]:
yarn_grew

{'nodes': {'vr1': {'concept': 'representative', 'type': 'V', 'var': 'vr1'},
  'vc1': {'concept': 'client', 'type': 'V', 'var': 'vc1'},
  'vl1': {'pred': 'locate-01', 'type': 'V', 'var': 'vl1'},
  'vm1': {'concept': 'meeting', 'type': 'V', 'var': 'vm1'},
  'va1': {'concept': 'or', 'type': 'V', 'var': 'va1'},
  's1': {'event': 's1', 'var': 's1', 'type': 'S'},
  's1-temp': {'type': 'F', 'feat': 'temp', 'var': 's1-temp'},
  's1-quant': {'type': 'F', 'feat': 'quant', 'var': 's1-quant'},
  's1-num': {'type': 'F', 'feat': 'num', 'var': 's1-num'},
  's1-def': {'type': 'F', 'feat': 'def', 'var': 's1-def'},
  'e1': {'rel': 'ARG1', 'type': 'E', 'var': 'e1'},
  'e2': {'rel': 'ARGM-LOC', 'type': 'E', 'var': 'e2'},
  'e3': {'rel': 'op1', 'type': 'E', 'var': 'e3'},
  'e4': {'rel': 'op2', 'type': 'E', 'var': 'e4'},
  'l1': {'value': 'past', 'var': 'l1', 'type': 'L', 'feat': 'temp'},
  'l2': {'value': 'exists', 'var': 'l2', 'type': 'L', 'feat': 'quant'},
  'l3': {'value': 'singular', 'var': 'l3', 'type

# Preprocessing

In [410]:
# reifies domain, mod, and poss when they have incoming H edges
# TO DO: make temp an L edge always

grs_path = "grs/main.grs"
grs = GRS(grs_path)
yarn_grew = grs.apply(Graph(yarn_grew), strat='main').json_data()

In [411]:
# save as json
# with open('output.json', 'w') as f:
#     json.dump(yarn_grew, f)

# Chains

In [412]:
# def get_chains(yarn_grew):
#     nodes = yarn_grew['nodes']
#     edges = yarn_grew['edges']

#     # Find the first S node
#     start_node = next(
#         (node for node in nodes.values() if node['type'] == 'S'),
#         None
#     )
#     if start_node is None:
#         raise ValueError("No S node found in graph")

#     all_chains = []

#     def traverse(chain):
#         current = chain[-1]

#         # Stop and save chain if we've hit an S or V node (after the start, and not the same node)
#         if current['type'] in ('S', 'V') and len(chain) > 1 and current['var'] != chain[0]['var']:
#             all_chains.append(chain)
#             return

#         src = current['var']
#         found_any = False

#         for edge in edges:
#             if edge['src'] == src:
#                 found_any = True
#                 tar = edge['tar']
#                 next_node = nodes[tar]
#                 traverse(chain + [next_node])

#         if not found_any:
#                 end_type = current['type']
#                 if end_type not in ('S', 'V'):
#                     raise ValueError(
#                         f"Chain ended on a non-S/V node of type '{end_type}': {current}"
#                     )
#                 if len(chain) > 1:  # don't save single-node chains
#                     all_chains.append(chain)

#     # Start from the first S node
#     traverse([start_node])

#     # Also start from every V node as a chain root
#     for node in nodes.values():
#         if node['type'] == 'V':
#             traverse([node])

#     return all_chains

In [413]:
def get_S_descendants(yarn_grew):
    nodes = yarn_grew["nodes"]
    edges = yarn_grew["edges"]

    adj = {}
    for e in edges:
        adj.setdefault(e["src"], []).append(e["tar"])

    result = {}

    for node_id, node_data in nodes.items():
        if node_data.get("type") != "S":
            continue

        visited = set()
        stack = [node_id]
        reachable_V = set()

        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)

            current_type = nodes[current].get("type")

            if current_type in ["L", "H", "V"]:
                reachable_V.add(current)

            if current_type == "C": # or D...
                continue

            for neighbor in adj.get(current, []):
                if neighbor not in visited:
                    stack.append(neighbor)

        result[node_id] = list(reachable_V)

    return result

In [414]:
#needs to be addded to yarn2fol function

def propagate_s_node_information(yarn_grew, s_descendants):

    for s_node, descendants in s_descendants.items():
        for descendant in descendants:
            for node, feats in yarn_grew['nodes'].items():
                if node == descendant:
                    feats['event'] = s_node
    
    return yarn_grew

In [415]:
s_descendants = get_S_descendants(yarn_grew)
yarn_grew = propagate_s_node_information(yarn_grew, s_descendants)

# Build F and R

In [416]:
id2var = {}
def build_F(yarn_grew_graph, id2var, variables):

    def fresh_variable(base=None):
        if base is None:
            base = 'e'
        i = 0

        while True:
            variable = base if i == 0 else f"{base}{i}"
            if variable not in variables:
                variables.add(variable)
                break
            i += 1
        return variable

    F = []
    nodes = yarn_grew_graph['nodes']
    edges = yarn_grew_graph['edges']

    for node, feats in nodes.items():

        # S node quantifcation
        # includes V -> S (C edges)
        # no S -> S (D edges) for now 
        if feats['type'] == 'S':

            id2var[node] = feats['var']
            variables.add(feats['var'])

            for edge1 in edges:
                if edge1['tar'] == node:
                    src = edge1['src']

                    if nodes[src]['type'] == 'C':
                        for edge2 in edges:
                            if edge2['tar'] == src:
                                incoming = edge2['src']
                else:
                    incoming = None

            F.append({
                            'id':node,
                            'incoming': incoming,
                            'outgoing':None,
                            'S':feats['event'],
                            'type':"∃",
                            'variable': id2var[node],
                            'tar_label': 'S', # change to 'label'
                        })
            
        # quantification
        if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['quant', 'temp']:
            for edge in edges:
                if edge['src'] == node:
                    tar = edge['tar']

                    if nodes[tar]['type'] == 'V':
                        
                        edge_label = feats['value'] if feats['value'] else feats['feat'] # accepts unlabeled temp/quant edges for now

                        if feats['feat'] == 'quant':
                            if edge_label == 'exists':
                                type = '∃'
                            elif edge_label == 'forall':
                                type = '∀'
                            else:
                                type = f'Q_{edge_label}'
                        elif feats['feat'] == 'temp':
                            type = f'T_{edge_label}'

                        if 'pred' in nodes[tar]:
                            tar_label = nodes[tar]['pred']
                        else:
                            tar_label = nodes[tar]['concept']

                        if tar not in id2var:
                            variable = fresh_variable(base=tar_label[0])
                            id2var[tar] = variable
                        else:
                            raise AssertionError(f"Double quantification. Variable for {tar} already exists in id2var.")
                        
                        F.append({
                            'id':tar,
                            'incoming':node,
                            'outgoing':None,
                            'S':feats['event'],
                            'type':type, 
                            'variable': id2var[tar] if feats['feat'] in ['quant', 'temp'] else None,
                            'tar_label':tar_label, # change to 'label'
                        })
        
        # negation, modality, aspect
        if (feats['type'] == "L" or feats['type'] == 'H') and feats['feat'] in ['neg', 'modal', 'aspect']: #main_out!
            for edge1 in edges:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if nodes[tar]['type'] in ['V', 'L', 'H']:
                        
                        edge_label = feats['value'] if feats['value'] else feats['feat']

                        for edge2 in edges:
                            if edge2['tar'] == node:
                                src = edge2['src']

                        F.append({
                            'id':node,
                            'incoming':src,
                            'outgoing':tar,
                            'S':feats['event'],
                            'type': edge_label,
                            'variable': None,
                            'tar_label': None, # change to 'label'
                        })
        
        # Any none-quantified V node is a constant
    for node, feats in nodes.items():
        if feats['type'] == 'V' and node not in id2var:
            if 'pred' in feats:
                id2var[node] = feats['pred'].upper()
            else:
                id2var[node] = feats['concept'].upper()

    F = [f for f in F if f['type'] not in ['perfective', 'state']]

    return F

In [417]:
F= build_F(yarn_grew, id2var=id2var, variables=set())
F

[{'id': 's1',
  'incoming': None,
  'outgoing': None,
  'S': 's1',
  'type': '∃',
  'variable': 's1',
  'tar_label': 'S'},
 {'id': 'vl1',
  'incoming': 'l1',
  'outgoing': None,
  'S': 's1',
  'type': 'T_past',
  'variable': 'l',
  'tar_label': 'locate-01'},
 {'id': 'vm1',
  'incoming': 'l2',
  'outgoing': None,
  'S': 's1',
  'type': '∃',
  'variable': 'm',
  'tar_label': 'meeting'},
 {'id': 'vr1',
  'incoming': 'h1',
  'outgoing': None,
  'S': 's1',
  'type': '∀',
  'variable': 'r',
  'tar_label': 'representative'},
 {'id': 'vc1',
  'incoming': 'h2',
  'outgoing': None,
  'S': 's1',
  'type': '∀',
  'variable': 'c',
  'tar_label': 'client'}]

In [418]:
def build_R(yarn_grew, id2var):

    nodes = yarn_grew['nodes']
    edges = yarn_grew['edges']
    
    R = {}
    for node, feats in nodes.items():
        if feats['type'] == "E":
            edge_label = feats['rel']

            for edge1 in edges:
                if edge1['tar'] == node:

                    src = edge1['src']

                    for edge2 in edges:
                        if edge2['src'] == node:
                            tar = edge2['tar']

                            if id2var[src].isupper(): # if constant
                                if tar not in R:
                                    R[tar] = [(edge_label, src, tar)]
                                else:
                                    R[tar].append((edge_label, src, tar))
                            else:
                                if src not in R:
                                    R[src] = [(edge_label, src, tar)]
                                else:
                                    R[src].append((edge_label, src, tar))
        
        if feats['type'] == "L" and feats['feat'] == 'num' and feats['value'] == 'plural': 
            for edge1 in edges:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if nodes[tar]['type'] == 'V':
                        
                        if tar not in R:
                            R[tar] = [('plural', tar)] 
                        else:
                            R[tar].append(('plural', tar))
        
        if feats['type'] == "L" and feats['feat'] == 'def': 
            for edge1 in edges:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if nodes[tar]['type'] == 'V':
                        
                        if tar not in R:
                            R[tar] = [('C', tar)]
                        else:
                            R[tar].append(('C', tar))

        if feats['type'] == "C":
            edge_label = feats['rel']

            for edge1 in edges:
                if edge1['tar'] == node:
                    src = edge1['src']

                    for edge2 in edges:
                        if edge2['src'] == node:
                            tar = edge2['tar']
                    
                            if tar not in R:
                                R[tar] = [(edge_label, src, tar)]
                            else:
                                R[tar].append((edge_label, src, tar))

        # S node modifiers
        # update as you annotate if you find trickier cases
            
        if feats['type'] == "L" and feats['feat'] in ['manner', 'loc', 'dir', 'duration', 'mod', 'freq']: 
            for edge1 in edges:
                if edge1['src'] == node:
                    tar = edge1['tar']

                    if nodes[tar]['type'] == 'V':
                        
                        edge_label = feats['value'] if feats['value'] else feats['feat']

                        # if 'pred' in nodes[tar]:
                        #     tar_label = nodes[tar]['pred']
                        # else:
                        #     tar_label = nodes[tar]['concept']

                        for edge2 in edges:
                            if edge2['tar'] == node:
                                src = edge2['src'].split('-')[0]
                            
                                if src not in R:
                                    R[src] = [(edge_label, src, tar)]
                                else:
                                    R[src].append((edge_label, src, tar))
    
    return R

In [419]:
def add_to_R(R, key, connective, relations):
    if key not in R:
        R[key] = {'and': [], 'or': []}
    R[key][connective].extend(relations)

In [420]:
from collections import defaultdict

def build_R(yarn_grew, id2var):

    nodes = yarn_grew['nodes']
    edges = yarn_grew['edges']

    # pre-index edges
    src_to_tars = defaultdict(list)
    tar_to_srcs = defaultdict(list)
    for edge in edges:
        src_to_tars[edge['src']].append(edge['tar'])
        tar_to_srcs[edge['tar']].append(edge['src'])
    
    R = {}
    for node, feats in nodes.items():
        if feats['type'] == "E":
            if feats['rel'].startswith('op'):
                continue

            edge_label = feats['rel']
            for src in tar_to_srcs[node]:
                for tar in src_to_tars[node]:
                    key = tar if id2var[src].isupper() else src
                    if nodes[tar]['concept'] == 'or':
                        grandchildren = [src_to_tars[e][0] for e in src_to_tars[tar]]
                        add_to_R(R, key, 'or', [(edge_label, src, t) for t in grandchildren])
                    elif nodes[tar]['concept'] == 'and':
                        grandchildren = [src_to_tars[e][0] for e in src_to_tars[tar]]
                        add_to_R(R, key, 'and', [(edge_label, src, t) for t in grandchildren])
                    else:
                        add_to_R(R, key, 'and', [(edge_label, src, tar)])

        if feats['type'] == "L" and feats['feat'] == 'num' and feats['value'] == 'plural':
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    add_to_R(R, tar, 'and', [('plural', tar)])

        if feats['type'] == "L" and feats['feat'] == 'def' and feats['value'] == 'definite':
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    add_to_R(R, tar, 'and', [('C', tar)])

        if feats['type'] == "C":
            edge_label = feats['rel']
            for src in tar_to_srcs[node]:
                for tar in src_to_tars[node]:
                    add_to_R(R, tar, 'and', [(edge_label, src, tar)])

        if feats['type'] == "L" and feats['feat'] in ['manner', 'loc', 'dir', 'duration', 'mod', 'freq']:
            for tar in src_to_tars[node]:
                if nodes[tar]['type'] == 'V':
                    edge_label = feats['value'] if feats['value'] else feats['feat']
                    for src in tar_to_srcs[node]:
                        src = src.split('-')[0]
                        add_to_R(R, src, 'and', [(edge_label, src, tar)])
    return R

In [422]:
yarn_grew

{'nodes': {'vr1': {'concept': 'representative',
   'type': 'V',
   'var': 'vr1',
   'event': 's1'},
  'vc1': {'concept': 'client', 'type': 'V', 'var': 'vc1', 'event': 's1'},
  'vl1': {'pred': 'locate-01', 'type': 'V', 'var': 'vl1', 'event': 's1'},
  'vm1': {'concept': 'meeting', 'type': 'V', 'var': 'vm1', 'event': 's1'},
  'va1': {'concept': 'or', 'type': 'V', 'var': 'va1', 'event': 's1'},
  's1': {'event': 's1', 'type': 'S', 'var': 's1'},
  's1-temp': {'feat': 'temp', 'type': 'F', 'var': 's1-temp'},
  's1-quant': {'feat': 'quant', 'type': 'F', 'var': 's1-quant'},
  's1-num': {'feat': 'num', 'type': 'F', 'var': 's1-num'},
  's1-def': {'feat': 'def', 'type': 'F', 'var': 's1-def'},
  'e1': {'rel': 'ARG1', 'type': 'E', 'var': 'e1'},
  'e2': {'rel': 'ARGM-LOC', 'type': 'E', 'var': 'e2'},
  'e3': {'rel': 'op1', 'type': 'E', 'var': 'e3'},
  'e4': {'rel': 'op2', 'type': 'E', 'var': 'e4'},
  'l1': {'feat': 'temp',
   'type': 'L',
   'value': 'past',
   'var': 'l1',
   'event': 's1'},
  'l2': {

In [424]:
R = build_R(yarn_grew, id2var)
R

{'vl1': {'and': [('ARGM-LOC', 'vl1', 'vm1')],
  'or': [('ARG1', 'vl1', 'vc1'), ('ARG1', 'vl1', 'vr1')]},
 'vm1': {'and': [('C', 'vm1')], 'or': []}}

# Create the Forest

In [450]:
def build_scope_forest(F, R, id2var):

    forest = {'nodes':{}, 'edges':[]}

    for i, f in enumerate(F):
        if f['id'] in R:
            and_rels = [
                f"{rel[0]}({id2var[rel[1]]},{id2var[rel[2]]})" if len(rel) == 3
                else f"{rel[0]}({id2var[rel[1]]})"
                for rel in R[f['id']]['and']
            ]
            or_rels = [
                f"{rel[0]}({id2var[rel[1]]},{id2var[rel[2]]})" if len(rel) == 3
                else f"{rel[0]}({id2var[rel[1]]})"
                for rel in R[f['id']]['or']
            ]
            relations = and_rels + [f"({' ∨ '.join(or_rels)})"] if or_rels else and_rels
        else:
            relations = []

        forest['nodes'][i] = {
            'id': f['id'],
            'incoming': f['incoming'],
            'outgoing': f['outgoing'],
            'S': f['S'],
            'type': f['type'],
            'variable': f['variable'],
            'tar_label': f['tar_label'],
            'relations': relations,
        }

    for k1, v1 in forest['nodes'].items(): # encode specified scope
        for k2, v2 in forest['nodes'].items():
            if v1['id'] == v2['incoming']:
                forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})
            if v1['id'] == v2['outgoing']:
                forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})
            if v1['outgoing'] and v2['incoming'] and v1['outgoing'] == v2['incoming']: # not sure this is smart
                forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})

    return forest

In [451]:
forest = build_scope_forest(F, R, id2var)
forest

{'nodes': {0: {'id': 's1',
   'incoming': None,
   'outgoing': None,
   'S': 's1',
   'type': '∃',
   'variable': 's1',
   'tar_label': 'S',
   'relations': []},
  1: {'id': 'vl1',
   'incoming': 'l1',
   'outgoing': None,
   'S': 's1',
   'type': 'T_past',
   'variable': 'l',
   'tar_label': 'locate-01',
   'relations': ['ARGM-LOC(l,m)', '(ARG1(l,c) ∨ ARG1(l,r))']},
  2: {'id': 'vm1',
   'incoming': 'l2',
   'outgoing': None,
   'S': 's1',
   'type': '∃',
   'variable': 'm',
   'tar_label': 'meeting',
   'relations': ['C(m)']},
  3: {'id': 'vr1',
   'incoming': 'h1',
   'outgoing': None,
   'S': 's1',
   'type': '∀',
   'variable': 'r',
   'tar_label': 'representative',
   'relations': []},
  4: {'id': 'vc1',
   'incoming': 'h2',
   'outgoing': None,
   'S': 's1',
   'type': '∀',
   'variable': 'c',
   'tar_label': 'client',
   'relations': []}},
 'edges': []}

## Add the Participant before Event constraint

In [432]:
# Predicates are introduced after their arguments (E relations only)
# C relations are encoded already in the Forest building forest

def add_participants_before_event_principle(forest, yarn_grew, R):

    for _, rels in R.items():
        for rel in rels['and'] + rels['or']:
            if len(rel) == 3:
                src = rel[1]
                tar = rel[2]

                for k1, v1 in forest['nodes'].items():
                    for k2, v2 in forest['nodes'].items():
                        if v1['id'] == src and v2['id'] == tar and \
                            yarn_grew['nodes'][src]['type'] == 'V' and \
                            yarn_grew['nodes'][tar]['type'] == 'V':
                            
                            forest['edges'].append({'src':k2, 'rel':'', 'tar':k1})
    
    return forest

In [433]:
def add_s_node_scope(forest, s_descendants):
    for k1, v1 in forest['nodes'].items():
        if v1['id'] in s_descendants:
            for k2, v2 in forest['nodes'].items():
                if v2['id'] in s_descendants[v1['id']]:
                    forest['edges'].append({'src':k1, 'rel':'', 'tar':k2})
                    
    return forest

# Get All Possible Trees

In [434]:
def get_all_possible_trees(n):
    nodes = list(range(n))
    for seq in itertools.product(nodes, repeat=n-2):
        yield nx.from_prufer_sequence(seq)

In [435]:
def get_all_possible_rooted_directed_trees(forest): # no need for 'rooted'

    nodes = list(forest['nodes'].keys())
    n_nodes = len(nodes)

    all_possible_rooted_directed_tree_edges = []

    for tree in get_all_possible_trees(n_nodes):

        for root in nodes:

            visited = set([root])
            stack = [root]
            directed_edges = []

            while stack:
                current = stack.pop()

                for neighbor in tree.neighbors(current):
                    if neighbor not in visited:
                        visited.add(neighbor)
                        stack.append(neighbor)

                        directed_edges.append({'src': current,'tar': neighbor})

            all_possible_rooted_directed_tree_edges.append({'edges': directed_edges})
    
    return all_possible_rooted_directed_tree_edges

# Build T_all

In [436]:
# Gets the children of nodes that don't introduce variables
def get_H_children(graph):
    H_children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        if not graph['nodes'][src]['variable']:
            H_children_dict[src] = H_children_dict.get(src, []) + [tar]

    return H_children_dict

In [437]:
# Get all children
def get_children(graph):
    children_dict = {}
    for edge in graph['edges']:
        src = edge['src']
        tar = edge['tar']
        children_dict[src] = children_dict.get(src, []) + [tar]
    return children_dict

# Extend children to descendants
def get_descendants(node, children_dict):
    descendants = []
    for child in children_dict.get(node, []):
        descendants.append(child)
        descendants.extend(get_descendants(child, children_dict))

    return descendants

def get_all_descendants(graph):
    children_dict = get_children(graph)
    descendants_dict = {}
    for node in children_dict:
        descendants_dict[node] = get_descendants(node, children_dict)

    return descendants_dict

In [438]:
# Constraint Checkers

def check_compatibility_of_scopes(tree, forest):
    descendants_tree = get_all_descendants(tree)
    descendants_forest = get_all_descendants(forest)

    for k, v in descendants_forest.items():
        if not v:
            continue
        if k not in descendants_tree:
            return False
        for descendant in v:
            if descendant not in descendants_tree[k]:
                return False
    return True

def check_locality_of_features(tree, forest):
    children_tree = get_children(tree)
    H_children_forest = get_H_children(forest)
    
    for k, v in H_children_forest.items():
        if not v:
            continue
        if k not in children_tree:
            return False
        for child in v:
            if child not in children_tree[k]:
                return False
    return True

In [439]:
def build_T_all(forest, valid_tree_edges):

    T_all = []
    for tree in valid_tree_edges:
        new_tree = forest.copy()
        new_tree['edges'] = tree['edges']
        T_all.append(new_tree)
    
    return T_all

## Reformat T_all

In [440]:
def reformat(graph):
    all_targets = {edge["tar"] for edge in graph["edges"]}
    root_id = next(nid for nid in graph["nodes"] if nid not in all_targets)
    
    def build(node_id):
        node = dict(graph["nodes"][node_id])
        node["children"] = [build(edge["tar"]) for edge in graph["edges"] if edge["src"] == node_id]
        return node
    
    return build(root_id)

# Interpretation

In [441]:
def conj(parts):
    parts = [p for p in parts if p and p.strip()]
    return " ∧ ".join(parts)


def wrap_quant(q, var, head, relations, bodies, connective="∧"):
    rel = conj(relations)
    head_part = f"{head}({var})"
    left = f"{head_part} ∧ {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}{var}. ( {left}\n {connective} ({body}) )"
    else:
        return f"{q}{var}. ( {left} )"
    
def wrap_generalized_quant(q, var, gen_quant, head, relations, bodies, connective="∧"):
    rel = conj(relations)
    head_part = f"{head}({var}) ∧ {gen_quant}({var})"
    left = f"{head_part} ∧ {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}{var}. ( {left}\n {connective} ({body}) )"
    else:
        return f"{q}{var}. ( {left} )"
    
def wrap_temp(q, var, S, head, relations, bodies, connective="∧"):
    rel = conj(relations)
    head_part = f"{head}({var},{S})"
    left = f"{head_part} ∧ {rel}" if rel else head_part
    body = conj(bodies)
    if body:
        return f"{q}{var}. ( {left}\n {connective} ({body}) )"
    else:
        return f"{q}{var}. ( {left} )"

def clean_formula(formula):
    clean_formula = formula.replace(" ∧ ()", "")
    return clean_formula

In [442]:
def interpret(root, temp_variable):
    
    if root is None:
        return ""

    child_formulas = [interpret(child, temp_variable) for child in root["children"]]

    if root["type"] == "∃":
        return wrap_quant("∃", root["variable"], root["tar_label"], root["relations"], child_formulas, connective="∧")
    
    if root["type"] == "∀":
        return wrap_quant("∀", root["variable"], root["tar_label"], root["relations"], child_formulas, connective="→")
    
    if root["type"].startswith("Q_"):
        gen_quant = root["type"][2:]
        return wrap_generalized_quant("∃", root["variable"], gen_quant, root["tar_label"], root["relations"], child_formulas, connective="∧")
    
    if root["type"] == "T_present":
        temp = f"{root['variable']}_O_{temp_variable}"
        child_formulas = [interpret(child, root["variable"]) for child in root["children"]]
        return wrap_temp("∃", root["variable"], root['S'], root["tar_label"], root["relations"] + [temp], child_formulas, connective="∧")

    if root["type"] == "T_past":
        temp = f"{root['variable']}≺{temp_variable}"
        child_formulas = [interpret(child, root["variable"]) for child in root["children"]]
        return wrap_temp("∃", root["variable"], root['S'], root["tar_label"], root["relations"] + [temp], child_formulas, connective="∧")
    
    if root["type"] == "T_future":
        temp = f"{temp_variable}≺{root['variable']}"
        child_formulas = [interpret(child, root["variable"]) for child in root["children"]]
        return wrap_temp("∃", root["variable"], root['S'], root["tar_label"], root["relations"] + [temp], child_formulas, connective="∧")
    
    if root["type"] == "neg":
        return f"¬ ( {conj(child_formulas)} )"
    
    if root["type"] == "possibility":
        return f"◇ ( {conj(child_formulas)} )"

    if root["type"] == "necessity":
        return f"□ ( {conj(child_formulas)} )"

In [443]:
def extract_number(path):

    match = re.match(r"(\d+)", path.name)
    return int(match.group(1)) if match else float("inf")


def load_yarn(input_path, recursive=False):
    if isinstance(input_path, (str, Path)):
        input_path = [input_path]

    all_files = []

    for path in input_path:
        path = Path(path)

        if path.is_file():
            if path.name.endswith(".yarn.json"):
                all_files.append(path)

        elif path.is_dir():
            files = path.rglob("*.yarn.json") if recursive else path.glob("*.yarn.json")
            all_files.extend(files)

        else:
            raise FileNotFoundError(f"{path} does not exist")

    all_files = sorted(all_files, key=extract_number)

    graphs = []
    for file_path in all_files:
        with open(file_path, "r", encoding="utf-8") as f:
            graph = json.load(f)
            if graph.get('labels'):  # safer
                graphs.append((file_path, graph))

    return graphs

In [444]:
import traceback
import multiprocessing as mp


TIMEOUT = 20


def process_one(path, yarn_graph_json, output_queue):
    try:
        id2var = {}
        variables = set()

        yarn_graph = YARNGraph(yarn_graph_json)
        yarn_grew = yarn_graph.grew()
        yarn_grew = grs.apply(Graph(yarn_grew), strat='main').json_data()
        
        s_descendants = get_S_descendants(yarn_grew)
        yarn_grew = propagate_s_node_information(yarn_grew, s_descendants)

        # save as json
        # with open('output.json', 'w') as f:
        #     json.dump(yarn_grew, f)

        F = build_F(yarn_grew, id2var, variables)
        R = build_R(yarn_grew, id2var)
        forest = build_scope_forest(F, R, id2var)
        
        forest = add_participants_before_event_principle(forest, yarn_grew, R)
        forest = add_s_node_scope(forest, s_descendants)
        print(forest)
        
        all_possible_rooted_directed_tree_edges = get_all_possible_rooted_directed_trees(forest)

        valid_tree_edges = [
            tree for tree in all_possible_rooted_directed_tree_edges
            if check_locality_of_features(tree, forest)
        ]
        valid_tree_edges = [
            tree for tree in valid_tree_edges
            if check_compatibility_of_scopes(tree, forest)
        ]

        T_all = build_T_all(forest, valid_tree_edges)
        T_all = [reformat(tree) for tree in T_all]

        results = []
        for T in T_all:
            results.append(clean_formula(interpret(T, 'NOW')))

        output_queue.put({
            "path": str(path),
            "meta": yarn_graph_json.get("meta", {}),
            "results": results,
            "error": None
        })

    except Exception as e:
        output_queue.put({
            "path": str(path),
            "meta": yarn_graph_json.get("meta", {}),
            "results": None,
            "error": traceback.format_exc()
        })

def yarn2fol(yarn_graphs):
    for path, yarn_graph_json in yarn_graphs:

        print("\nProcessing:", path)

        output_queue = mp.Queue()
        p = mp.Process(target=process_one, args=(path, yarn_graph_json, output_queue))

        p.start()
        p.join(TIMEOUT)

        if p.is_alive():
            p.terminate()
            p.join()

            print(path)
            print("TIMEOUT after", TIMEOUT, "seconds")
            continue

        if output_queue.empty():
            print(path)
            print("No output returned")
            continue

        result = output_queue.get()

        if result["error"]:
            print(path)
            print("ERROR:")
            print(result["error"])
            continue

        print(result["path"])
        print(result["meta"].get("type"), ":", result["meta"].get("text"))

        for r in result["results"]:
            print(r)
            print("---")

In [445]:
FOLDER_PATH = "annotations/FRACAS_1premise_yesno/"
FILE = "89p.yarn.json"

corpus = load_yarn(FOLDER_PATH+FILE)
yarn2fol(corpus)


Processing: annotations/FRACAS_1premise_yesno/89p.yarn.json
{'nodes': {0: {'id': 's1', 'incoming': None, 'outgoing': None, 'S': 's1', 'type': '∃', 'variable': 's1', 'tar_label': 'S', 'relations': []}, 1: {'id': 'vl1', 'incoming': 'l1', 'outgoing': None, 'S': 's1', 'type': 'T_past', 'variable': 'l', 'tar_label': 'locate-01', 'relations': ['ARGM-LOC(l,m)', '(ARG1(l,c) ∨ ARG1(l,r))']}, 2: {'id': 'vm1', 'incoming': 'l2', 'outgoing': None, 'S': 's1', 'type': '∃', 'variable': 'm', 'tar_label': 'meeting', 'relations': ['C(m)']}, 3: {'id': 'vr1', 'incoming': 'h1', 'outgoing': None, 'S': 's1', 'type': '∀', 'variable': 'r', 'tar_label': 'representative', 'relations': []}, 4: {'id': 'vc1', 'incoming': 'h2', 'outgoing': None, 'S': 's1', 'type': '∀', 'variable': 'c', 'tar_label': 'client', 'relations': []}}, 'edges': [{'src': 2, 'rel': '', 'tar': 1}, {'src': 4, 'rel': '', 'tar': 1}, {'src': 3, 'rel': '', 'tar': 1}, {'src': 0, 'rel': '', 'tar': 1}, {'src': 0, 'rel': '', 'tar': 2}, {'src': 0, 'rel':

In [446]:
# yarn2fol(load_yarn('annotations/1.yarn.json')) check why it doesn't work?

# Vampire

In [447]:
def conj_tptp(parts):
    parts = [p for p in parts if p and p.strip()]
    return " & ".join(parts)

def wrap_quant_tptp(q, var, head, relations, body, connective):
    rel = conj_tptp(relations)
    head_part = f"{head}({var})"

    if rel:
        left = f"{head_part} & {rel}"
    else:
        left = head_part

    return f"{q} [{var}] : ( {left} {connective} ( {body} ) )"

def clean_formula_tptp(formula):
    clean_formula = formula.replace(" & (  )", "").replace("-", "_")
    return clean_formula

In [448]:
def interpret_vampire(root, temp_variable):
    
    if root is None:
        return ""
    
    if root["type"] == "∃_s" or root["type"] == "Q_exists":
        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )

    if root["type"] == "Q_forall":
        return wrap_quant_tptp(
            "!",
            root["variable"],
            root["tar_label"],
            root["relations"],
            interpret_vampire(root["child"], temp_variable),
            "=>"
        )

    if root["type"] == "Q_present":
        temp = f"{root['variable']}_O_{temp_variable}"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )

    if root["type"] == "Q_past":
        temp = f"before({root['variable']},{temp_variable})"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )
    
    if root["type"] == "Q_future":
        temp = f"before({temp_variable},{root['variable']})"

        return wrap_quant_tptp(
            "?",
            root["variable"],
            root["tar_label"],
            root["relations"] + [temp],
            interpret_vampire(root["child"], temp_variable),
            "&"
        )
    
    if root["type"] == "Q_neg":
        return f"~( {interpret_vampire(root['child'], temp_variable)} )"

In [449]:
print(clean_formula_tptp(interpret_vampire(T_all[0], "NOW")))

NameError: name 'T_all' is not defined